In [0]:
# Install required library to read Excel files
%pip install openpyxl

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Using df_memb only — RFM requires known customer identity
df_memb = pd.read_parquet("/Volumes/workspace/default/retail_data/df_memb.parquet")

# DATASET INTEGRITY CHECK
# Validating transaction date boundaries before RFM calculation
# ---------------------------------------------------------------

# Find min and max transaction date for every month
monthly_date_range = df_memb.groupby(
    df_memb['InvoiceDate'].dt.to_period('M')
)['InvoiceDate'].agg(['min', 'max']).reset_index()

monthly_date_range.columns = ['YearMonth', 'First Transaction', 'Last Transaction']

print("TRANSACTION DATE RANGE PER MONTH")
print(monthly_date_range.to_string(index=False))


In [0]:
# Set analysis date — one day after the last transaction in the dataset
# This is used to calculate recency (how many days since last purchase)

# to calculate how many days ago each customer last purchased. 
# The last purchase date is "2011-12-09". 
# Using the day after avoids a zero recency score for the most recent customers.
analysis_date = pd.Timestamp('2011-12-10')

print(f"Data loaded — {len(df_memb):,} rows")

print(f"\nDataset starts: {df_memb['InvoiceDate'].min()}")
print(f"Dataset ends: {df_memb['InvoiceDate'].max()}")
print(f"Analysis date: {analysis_date.date()}")

In [0]:
# RFM CALCULATION
# For every customer we calculate:
    # Recency   — days since last purchase
    # Frequency — number of unique orders
    # TotalSpend — total amount spent
# -----------------------------------

rfm = df_memb.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (analysis_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    TotalSpend=('TotalRevenue', 'sum')
).reset_index()

print("RFM TABLE — FIRST 10 CUSTOMERS")
print(rfm.head(10).to_string(index=False))

print(f"\nRFM SUMMARY STATISTICS")
print(rfm[['Recency', 'Frequency', 'TotalSpend']].describe().round(2))

In [0]:
# RFM SCORING
# Every customer gets a score of 1-5 for each
# R, F and TotalSpend dimension
# 5 = best, 1 = worst
#
# Recency:   lower days = better = higher score
# Frequency: more orders = better = higher score
# TotalSpend: more spend = better = higher score
# -----------------------------------------------

# We use pd.qcut to split customers into 5 equal groups
# qcut splits by quantile — each group has equal number of customers

# RECENCY SCORE
# Note: for recency we reverse the score (5=lowest days, 1=highest days)
# because buying recently is BETTER than buying a long time ago
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=[5,4,3,2,1])

# FREQUENCY SCORE
# More orders = better = higher score
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])

# TOTALSPEND SCORE
# More spend = better = higher score
rfm['S_Score'] = pd.qcut(rfm['TotalSpend'], q=5, labels=[1,2,3,4,5])

# COMBINE INTO ONE RFM SCORE
# Concatenate the three scores into one string e.g. "555" or "123"
rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['S_Score'].astype(str)

# CALCULATE TOTAL SCORE (sum of all three)
rfm['Total_Score'] = rfm['R_Score'].astype(int) + rfm['F_Score'].astype(int) + rfm['S_Score'].astype(int)

print("RFM SCORES — FIRST 10 CUSTOMERS")
print(rfm[['Customer ID','Recency','Frequency','TotalSpend','R_Score','F_Score','S_Score','RFM_Score','Total_Score']].head(10).to_string(index=False))